In [ ]:
import matplotlib as plt

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Load and concatenate the 14 files
df_list = []
for week in range(1, 15):
    filename = f"cfb_punts_week{week}.csv"
    df_week = pd.read_csv(filename)
    df_list.append(df_week)

df = pd.concat(df_list)

# Calculate Success Rate
df['success'] = ((df['distance'] <= df['yards_gained']) |
                 ((df['down'] == 1) & (df['yards_gained'] >= 0.5 * df['distance'])) |
                 ((df['down'] == 2) & (df['yards_gained'] >= df['distance'])) |
                 ((df['down'] == 3) & (df['yards_gained'] >= 0.4 * df['distance'])) |
                 ((df['down'] == 4) & (df['yards_gained'] >= 0.7 * df['distance']))).astype(int)

# Calculate EPA
df['EPA'] = 0.3 * df['success'] + 0.5 * (df['yards_gained'] - 10) + 4.1 * (df['scoring'] == 'Touchdown') - 2.8 * (df['scoring'] == 'Interception') - 0.4 * (df['scoring'] == 'Sack')

# Output the results
print(df[['game_id', 'drive_id', 'play_number', 'EPA']])


       game_id     drive_id  play_number  EPA
0    401403853   4014038532            5 -5.0
1    401403853   4014038533            4 -5.0
2    401403853   4014038535            7 -5.0
3    401403853  40140385310            7 -4.5
4    401403853  40140385319            5 -5.0
..         ...          ...          ...  ...
141  401507088  40150708815            8 -5.0
142  401507088  40150708817            5 -5.0
143  401507088  40150708820            5 -5.0
144  401507088  40150708824           10 -4.0
145  401507088  40150708826            8  7.3

[12746 rows x 4 columns]


In [5]:
import pandas as pd
import re

# Load and concatenate the 14 files
df_list = []
for week in range(1, 15):
    filename = f"cfb_punts_week{week}.csv"
    df_week = pd.read_csv(filename)
    df_list.append(df_week)

df = pd.concat(df_list)

# Extract the punter's name from the 'play_text' column
df['player'] = df['play_text'].str.extract(r'^(.*?) punt', flags=re.IGNORECASE)

# Calculate Success Rate
df['success'] = ((df['distance'] <= df['yards_gained']) |
                 ((df['down'] == 1) & (df['yards_gained'] >= 0.5 * df['distance'])) |
                 ((df['down'] == 2) & (df['yards_gained'] >= df['distance'])) |
                 ((df['down'] == 3) & (df['yards_gained'] >= 0.4 * df['distance'])) |
                 ((df['down'] == 4) & (df['yards_gained'] >= 0.7 * df['distance']))).astype(int)

# Calculate EPA
df['EPA'] = 0.3 * df['success'] + 0.5 * (df['yards_gained'] - 10) + 4.1 * (df['scoring'] == 'Touchdown') - 2.8 * (df['scoring'] == 'Interception') - 0.4 * (df['scoring'] == 'Sack')

# Output the results
print(df[['game_id', 'drive_id', 'play_number', 'player', 'EPA']])


       game_id     drive_id  play_number           player  EPA
0    401403853   4014038532            5     Matt Hayball -5.0
1    401403853   4014038533            4  Matthew Shipley -5.0
2    401403853   4014038535            7  Matthew Shipley -5.0
3    401403853  40140385310            7  Matthew Shipley -4.5
4    401403853  40140385319            5  Matthew Shipley -5.0
..         ...          ...          ...              ...  ...
141  401507088  40150708815            8     Evan Matthes -5.0
142  401507088  40150708817            5     Josh Carlson -5.0
143  401507088  40150708820            5     Evan Matthes -5.0
144  401507088  40150708824           10     Evan Matthes -4.0
145  401507088  40150708826            8     Evan Matthes  7.3

[12746 rows x 5 columns]


In [19]:
import pandas as pd
import re

# Load and concatenate the 14 files
df_list = []
for week in range(1, 15):
    filename = f"cfb_punts_week{week}.csv"
    df_week = pd.read_csv(filename)
    df_list.append(df_week)

df = pd.concat(df_list)

# Extract the punter's name from the 'play_text' column
df['player'] = df['play_text'].str.extract(r'^(.*?) punt', flags=re.IGNORECASE)

# Calculate Success Rate
df['success'] = ((df['distance'] <= df['yards_gained']) |
                 ((df['down'] == 1) & (df['yards_gained'] >= 0.5 * df['distance'])) |
                 ((df['down'] == 2) & (df['yards_gained'] >= df['distance'])) |
                 ((df['down'] == 3) & (df['yards_gained'] >= 0.4 * df['distance'])) |
                 ((df['down'] == 4) & (df['yards_gained'] >= 0.7 * df['distance']))).astype(int)

# Calculate Expected EPA (pEPA)
df['pEPA'] = 0.3 * df['success'] - 0.5 * df['yard_line']

# Calculate EPA
df['EPA'] = df['pEPA'] + 0.5 * (df['yards_gained'] - 10) + 4.1 * (df['scoring'] == 'Touchdown') - 2.8 * (df['scoring'] == 'Interception') - 0.4 * (df['scoring'] == 'Sack')

# Rank players by total EPA
df_player_epa = df.groupby('player')['EPA', 'pEPA'].sum().reset_index()
df_player_epa = df_player_epa.sort_values(by='EPA', ascending=False)

# Output the results
print(df_player_epa)


                       player     EPA    pEPA
474         punt for no gain,    -7.0    -2.0
133            Corey Peterson    -7.0    -2.0
339         Michael Drotzmann   -12.5    -7.5
294             Lane McGregor   -14.7   -35.2
365         Noah Rauschenberg   -16.0   -11.0
436           Timmy Bleekrode   -17.7   -16.7
161           Dylan Van Dusen   -18.0   -13.0
260        Jose Romo-Martinez   -19.5   -14.5
452             Tyler Flippen   -20.0   -15.0
312               Luke Pawlak   -20.5   -15.5
425                Shane Hamm   -21.5   -16.5
187           Grayden Addison   -21.5   -16.5
62            Blake Stenstrom   -22.0   -17.0
282                Kevin Ryan   -22.0   -17.0
329            Matthew Rhodes   -22.5   -17.5
286            Kriston Esnard   -23.0   -18.0
303                Lucas Maue   -23.5   -18.5
350           Nathan De Bruin   -23.5   -18.5
7              Ah-Shaun Davis   -24.0   -19.0
457          Walker Bradberry   -24.0   -19.0
76                Bret Bushka   -2

C:\Users\RaymondCarpenter\AppData\Local\Temp\ipykernel_20240\2018817607.py:30: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df_player_epa = df.groupby('player')['EPA', 'pEPA'].sum().reset_index()
